In [11]:
import h5py
from itertools import product
import os
import numpy as np
import nibabel as nib
import seaborn as sns
import matplotlib.pyplot as plt
from statsmodels.stats.multitest import multipletests
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.formula.api import mixedlm

# 加载神经反应数据

# 加载行为数据

file_path1 = 'face10156_z_correlations_real_si.h5'

def load_data(file_path):
    """Function to load data from subarray_0 to subarray_399 from a given file path."""
    with h5py.File(file_path, 'r') as file:
        data_list = []
        for i in range(400):  # Load from subarray_0 to subarray_399
            dataset_name = f'z_corr_matrix_{i}'
            if dataset_name in file:
                data = file[dataset_name][:]
                data_list.append(data)
            else:
                print(f"Dataset '{dataset_name}' not found in the file.")
        return data_list

# Load data for each file
all_z_correlations_real = load_data(file_path1)

msub_r_list = []
for brain_area in range(400):
    z_corr_matrix_real = all_z_correlations_real[brain_area]
    corr_sametrial_stacked = []

    for matrix in z_corr_matrix_real:
        # Extract the diagonal of the submatrices
        submatrix = np.diag(matrix)
        # Append to the list for current brain area
        corr_sametrial_stacked.append(submatrix)

    # Append the list of stacked arrays to the main list
    msub_r_list.append(corr_sametrial_stacked)
    
realpair = np.array(msub_r_list)  # 假设数据保存为.npy文件


import h5py
from itertools import product
import os
import numpy as np
import nibabel as nib
import seaborn as sns
import matplotlib.pyplot as plt
from statsmodels.stats.multitest import multipletests

#Define file paths for saving
file_path6 = 'face10156_cross_correlations_si.h5'

def load_data1(file_path):
    """Function to load data from datasets with names general_s_i and general_r_i   
    (where i ranges from 0 to 399) from a given file path, and store them in separate lists."""
    
    data1_list = []  # List to store data from general_s_ datasets
    data2_list = []  # List to store data from general_r_ datasets
    data3_list = []  # List to store data from general_s_ datasets
    data4_list = []  # List to store data from general_r_ datasets

    with h5py.File(file_path, 'r') as file:
        for i in range(400):  # Iterate over indices from 0 to 399
            s1_dataset_name = f'all_correlations_sarb_{i}'
            r1_dataset_name = f'all_correlations_rbsa_{i}'
            s2_dataset_name = f'all_correlations_sbra_{i}'
            r2_dataset_name = f'all_correlations_rasb_{i}'

            # Check and read general_s_ dataset
            if s1_dataset_name in file:
                data1 = file[s1_dataset_name][:]
                data1_list.append(data1)
            else:
                print(f"Dataset '{s1_dataset_name}' not found in the file.")

            # Check and read general_r_ dataset
            if r1_dataset_name in file:
                data2 = file[r1_dataset_name][:]
                data2_list.append(data2)
            else:
                print(f"Dataset '{r1_dataset_name}' not found in the file.")

            # Check and read general_s_ dataset
            if s2_dataset_name in file:
                data3 = file[s2_dataset_name][:]
                data3_list.append(data3)
            else:
                print(f"Dataset '{s2_dataset_name}' not found in the file.")

            # Check and read general_r_ dataset
            if r2_dataset_name in file:
                data4 = file[r2_dataset_name][:]
                data4_list.append(data4)
            else:
                print(f"Dataset '{r2_dataset_name}' not found in the file.")
        return data1_list, data2_list, data3_list, data4_list

# Load data for each file
s1,r1,s2,r2 = load_data1(file_path6)

crosspairs=[]

for brain_area in range(400):
    # 对于每个脑区，提取相应的数据
    s1data = s1[brain_area]  # shape (23, 24, number_of_voxels)
    r1data = r1[brain_area]  # shape (23, 24, number_of_voxels)# 创建两个 400x23x22x22x22 的相似性矩阵
    s2data = s2[brain_area]  # shape (23, 24, number_of_voxels)
    r2data = r2[brain_area]  # shape (23, 24, number_of_voxels)# 创建两个 400x23x22x22x22 的相似性矩阵

    crosspaira=[]
    crosspairb=[]

    for sub in range(23):
        sub_s1data=np.mean(s1data[sub],axis=0)
        sub_r1data=np.mean(r1data[sub],axis=0)
        sub_s2data=np.mean(s2data[sub],axis=0)
        sub_r2data=np.mean(r2data[sub],axis=0)
        sa=(sub_s1data+sub_r1data)/2
        sb=(sub_s2data+sub_r2data)/2
        crosspaira.append(np.diag(sa))
        crosspairb.append(np.diag(sb))

    crosspair=np.vstack((crosspaira,crosspairb))
    crosspairs.append(crosspair)

In [12]:
crosspairs=np.array(crosspairs)
realpair=np.array(realpair)

In [22]:
import numpy as np
import pandas as pd
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
import multiprocessing as mp
from functools import partial
import os

# 启用 Pandas 与 R DataFrame 互操作
pandas2ri.activate()

# 定义处理单个脑区的函数
def process_region(region, realpair, crosspairs):
    import pandas as pd
    import numpy as np
    import os
    from rpy2.robjects import pandas2ri
    import rpy2.robjects as ro
    pandas2ri.activate()

    process_id = os.getpid()
    print(f"Process {process_id} - Processing region {region}")

    try:
        # === 1. 构造配对数据，每行表示一个 trial-pair ===
        data_list = []
        for subject in range(46):
            for t in range(24):
                condition = 1 if t < 12 else 2
                stimulus_group = 2 if subject >= 23 else 1
                y_real = realpair[region, subject, t]
                y_pseudo = crosspairs[region, subject, t]
                data_list.append([subject, t, condition, y_real, y_pseudo, stimulus_group])

        df_pairs = pd.DataFrame(data_list, columns=["Subject", "Trial", "Condition", "y_real", "y_pseudo", "StimulusGroup"])

        df_pairs["z_real"] = df_pairs.groupby("Subject")["y_real"].transform(lambda x: (x - x.mean()) / x.std(ddof=0))
        df_pairs["z_pseudo"] = df_pairs.groupby("Subject")["y_pseudo"].transform(lambda x: (x - x.mean()) / x.std(ddof=0))

        # === 3. 剔除任意一侧为离群值的 pair ===
        df_pairs["is_outlier"] = (df_pairs["z_real"].abs() > 3) | (df_pairs["z_pseudo"].abs() > 3)
        df_pairs_clean = df_pairs[~df_pairs["is_outlier"]].copy()

        if df_pairs_clean.empty:
            print(f"Process {process_id} - Skipping region {region}: all data removed after cleaning")
            return None

        # === 4. 展开为 lmer 所需格式 ===
        df_long = pd.concat([
            df_pairs_clean.assign(PairType=1, y=df_pairs_clean["y_real"]),
            df_pairs_clean.assign(PairType=0, y=df_pairs_clean["y_pseudo"])
        ], ignore_index=True)

        df_long = df_long[["Subject", "Trial", "Condition", "StimulusGroup", "PairType", "y"]]
        df_long["Subject"] = df_long["Subject"].astype(str)
        df_long["Trial"] = df_long["Trial"].astype(str)

        df_long["y"] = df_long["y"] * 10  # 放大避免数值精度问题

        # === 5. 转换为 R dataframe ===
        r_df = pandas2ri.py2rpy(df_long)
        ro.globalenv["df"] = r_df

        # === 6. R代码：拟合 lmer + permutation ===
        r_code = """
        library(lme4)
        df$PairType <- as.factor(df$PairType)
        df$Condition <- as.factor(df$Condition)
        df$StimulusGroup <- as.factor(df$StimulusGroup)
        df$Subject <- as.factor(df$Subject)
        df$Trial <- as.factor(df$Trial)

        model_full <- lmer(y ~ Condition + PairType + StimulusGroup + (1 | Subject), data=df, REML=FALSE)

        singular <- isSingular(model_full)
        if (singular) warning("Singular fit detected")

        real_pairtype_coef <- fixef(model_full)["PairType1"]

        # === PairType 固定效应的 95% Wald 置信区间(来自真模型,与 permutation 无关) ===
        ci <- confint(model_full, parm="PairType1", method="Wald")
        ci_low  <- as.numeric(ci[1])
        ci_high <- as.numeric(ci[2])

        N_PERMUTATIONS <- 3000
        perm_pairtype_coefs <- numeric(N_PERMUTATIONS)
        fail_count <- 0

        for (i in 1:N_PERMUTATIONS) {
            perm_df <- df
            for (subj in unique(df$Subject)) {
                for (cond in unique(df$Condition)) {
                    for (grp in unique(df$StimulusGroup)) {
                        idx <- which(df$Subject == subj & df$Condition == cond & df$StimulusGroup == grp)
                        if (length(idx) > 1) {
                            perm_df$PairType[idx] <- sample(df$PairType[idx])
                        }
                    }
                }
            }
            perm_df$PairType <- factor(perm_df$PairType, levels=c("0", "1"))
            perm_model <- tryCatch(
                lmer(y ~ Condition + PairType + StimulusGroup + (1 | Subject), data=perm_df, REML=FALSE),
                error = function(e) { fail_count <<- fail_count + 1; return(NA) }
            )
            if (is.na(perm_model)[1]) next
            perm_pairtype_coefs[i] <- fixef(perm_model)["PairType1"]
        }

        perm_pairtype_coefs <- perm_pairtype_coefs[!is.na(perm_pairtype_coefs)]
        p_value_pairtype_perm <- mean(perm_pairtype_coefs >= real_pairtype_coef)

        list(
            intercept=as.numeric(fixef(model_full)["(Intercept)"]),
            pair=as.numeric(fixef(model_full)["PairType1"]),
            condition2=as.numeric(fixef(model_full)["Condition2"]),
            stimgroup2=as.numeric(fixef(model_full)["StimulusGroup2"]),
            sigma_resid=as.numeric(sigma(model_full)),
            ci_low=ci_low,
            ci_high=ci_high,
            p_value_pairtype_perm=p_value_pairtype_perm,
            singular=singular,
            fail_count=fail_count
        )
        """

        r_results = ro.r(r_code)
        results = [region] + list(r_results)
        print(f"Process {process_id} - Completed region {region}")
        return results

    except Exception as e:
        print(f"Process {process_id} - Error processing region {region}: {e}")
        return None


# 主函数 - 并行处理
def run_analysis(realpair, crosspairs):
    # 设置并行处理的核心数
    num_cores = 10
    print(f"Running with {num_cores} cores")

    # 创建进程池
    pool = mp.Pool(processes=num_cores)

    # 创建偏函数，固定realpair和crosspairs参数
    process_func = partial(process_region, realpair=realpair, crosspairs=crosspairs)

    # 并行处理所有脑区 (测试单区用 range(1)，跑全部改回 range(400))
    results = pool.map(process_func, range(400))

    # 关闭进程池
    pool.close()
    pool.join()

    # 过滤掉None值
    results_list = [r for r in results if r is not None]

    # 创建结果DataFrame
    df_results = pd.DataFrame(results_list, columns=[
        "Brain_Region", "Intercept", "Pair_Effect",
        "Condition_Effect", "StimulusGroup_Effect", "sigma_resid",
        "PairType_CI_low", "PairType_CI_high",
        "PairType_p_perm", "Singular_Fit", "Fail_Count"
    ])

    # 计算 d_like 和原始尺度的 beta/CI(x10 在 d_like 比值中约掉；报原始尺度时除回10)
    df_results["d_like"] = df_results["Pair_Effect"] / df_results["sigma_resid"]
    df_results["PairType_beta_raw"] = df_results["Pair_Effect"] / 10.0
    df_results["PairType_CI_low_raw"] = df_results["PairType_CI_low"] / 10.0
    df_results["PairType_CI_high_raw"] = df_results["PairType_CI_high"] / 10.0

    # 保存结果
    #df_results.to_csv("real_vs_pseudo_results_global_perm_si.csv", index=False)

    return df_results

# 使用方法:
results = run_analysis(np.array(realpair), np.array(crosspairs))

Running with 10 cores
Process 326244 - Processing region 0
Process 326245 - Processing region 10
Process 326246 - Processing region 20
Process 326247 - Processing region 30
Process 326248 - Processing region 40
Process 326249 - Processing region 50
Process 326250 - Processing region 60
Process 326251 - Processing region 70
Process 326252 - Processing region 80
Process 326253 - Processing region 90


R[write to console]: In addition: 
R[write to console]: There were 50 or more warnings (use warnings() to see the first 50)
R[write to console]: 



Process 326247 - Completed region 30
Process 326247 - Processing region 31


R[write to console]: In addition: 
R[write to console]: There were 50 or more warnings (use warnings() to see the first 50)
R[write to console]: 



Process 326253 - Completed region 90
Process 326253 - Processing region 91


...

Process 326248 - Completed region 389


R[write to console]: In addition: 
R[write to console]: There were 50 or more warnings (use warnings() to see the first 50)
R[write to console]: 



Process 326245 - Completed region 379


R[write to console]: In addition: 
R[write to console]: There were 50 or more warnings (use warnings() to see the first 50)
R[write to console]: 



Process 326246 - Completed region 399


In [15]:
from statsmodels.stats.multitest import multipletests

# 进行 FDR 校正（对所有脑区的 Time_Condition_p 进行校正）
results["PairType_p_perm_FDR"] = multipletests(results["PairType_p_perm"], method="fdr_bh")[1]
significant_regions1 = results[
    (results["PairType_p_perm_FDR"] < 0.05)]

print(significant_regions1)
# 提取符合条件的脑区编号
significant_region_numbers = significant_regions1["Brain_Region"].tolist()

# 输出脑区编号
print(significant_region_numbers)

     Brain_Region                 Intercept            Pair_Effect  \
3               3     [-0.8512745761712716]  [0.37060990964602947]   
36             36     [-0.6391077260446814]  [0.31281662874671406]   
40             40     [-0.5796542351506541]  [0.34911912663916495]   
54             54     [-1.0293698420502573]  [0.24102008214759688]   
58             58     [-1.7261096470778294]  [0.25143861949670354]   
59             59    [0.003089334508161923]  [0.20083864658482312]   
76             76     [0.03898582201551462]   [0.1900492426164295]   
78             78     [0.17126783887239602]  [0.18757199977700173]   
79             79     [0.23765078635660364]   [0.2290666822929024]   
82             82  [-0.0033333365896475327]  [0.18177546440720582]   
100           100     [-0.1930058686979455]  [0.41447979612017866]   
106           106    [-0.04539568100435506]  [0.21702312604866578]   
139           139    [-0.14735729767131064]   [0.4028746395102664]   
140           140   

In [20]:
import numpy as np
import pandas as pd
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
import multiprocessing as mp
from functools import partial
import os

pandas2ri.activate()

# ======================================================================
# Robustness toggle
#   False -> MAIN analysis (identical to the original: trial-level |z|>3 only)
#   True  -> ROBUSTNESS analysis: additionally drop extreme SUBJECTS, defined
#            on the between-subject distribution of each subject's cross-trial
#            mean DSPS (|z|>3), applied to the true side OR the pseudo side.
#            The criterion is fixed in advance, independent of the
#            true-vs-pseudo difference, and applied per region.
# ======================================================================
EXCLUDE_EXTREME_SUBJECTS = True
SUBJECT_Z_THRESH = 3.0


def process_region(region, realpair, crosspairs,
                   exclude_extreme_subjects=False, subj_z_thresh=3.0):
    import pandas as pd
    import numpy as np
    import os
    from rpy2.robjects import pandas2ri
    import rpy2.robjects as ro
    pandas2ri.activate()

    process_id = os.getpid()
    print(f"Process {process_id} - Processing region {region}")

    try:
        # === 1. build pair data, one row per trial-pair ===
        data_list = []
        for subject in range(46):
            for t in range(24):
                condition = 1 if t < 12 else 2
                stimulus_group = 2 if subject >= 23 else 1
                y_real = realpair[region, subject, t]
                y_pseudo = crosspairs[region, subject, t]
                data_list.append([subject, t, condition, y_real, y_pseudo, stimulus_group])

        df_pairs = pd.DataFrame(data_list, columns=["Subject", "Trial", "Condition",
                                                    "y_real", "y_pseudo", "StimulusGroup"])

        # === 2. within-subject z (for trial-level outlier detection only) ===
        df_pairs["z_real"] = df_pairs.groupby("Subject")["y_real"].transform(
            lambda x: (x - x.mean()) / x.std(ddof=0))
        df_pairs["z_pseudo"] = df_pairs.groupby("Subject")["y_pseudo"].transform(
            lambda x: (x - x.mean()) / x.std(ddof=0))

        # === 3. trial-level outlier removal (either side |z|>3) ===
        df_pairs["is_outlier"] = (df_pairs["z_real"].abs() > 3) | (df_pairs["z_pseudo"].abs() > 3)
        df_pairs_clean = df_pairs[~df_pairs["is_outlier"]].copy()

        if df_pairs_clean.empty:
            print(f"Process {process_id} - Skipping region {region}: empty after trial cleaning")
            return None

        # === 3b. ROBUSTNESS: between-subject extreme-subject removal ===
        n_excluded_subjects = 0
        if exclude_extreme_subjects:
            subj_mean = df_pairs_clean.groupby("Subject").agg(
                rm=("y_real", "mean"), pm=("y_pseudo", "mean"))

            def _z(s):
                sd = s.std(ddof=0)
                return (s - s.mean()) / sd if sd > 0 else s * 0.0

            zr = _z(subj_mean["rm"])
            zp = _z(subj_mean["pm"])
            extreme = subj_mean.index[(zr.abs() > subj_z_thresh) | (zp.abs() > subj_z_thresh)]
            n_excluded_subjects = len(extreme)
            df_pairs_clean = df_pairs_clean[~df_pairs_clean["Subject"].isin(extreme)].copy()

            if df_pairs_clean["Subject"].nunique() < 3:
                print(f"Process {process_id} - Skipping region {region}: <3 subjects after subject removal")
                return None

        # === 4. long format for lmer ===
        df_long = pd.concat([
            df_pairs_clean.assign(PairType=1, y=df_pairs_clean["y_real"]),
            df_pairs_clean.assign(PairType=0, y=df_pairs_clean["y_pseudo"])
        ], ignore_index=True)

        df_long = df_long[["Subject", "Trial", "Condition", "StimulusGroup", "PairType", "y"]]
        df_long["Subject"] = df_long["Subject"].astype(str)
        df_long["Trial"] = df_long["Trial"].astype(str)
        df_long["y"] = df_long["y"] * 10

        ro.globalenv["df"] = pandas2ri.py2rpy(df_long)

        # === 5. lmer + permutation (unchanged) ===
        r_code = """
        library(lme4)
        df$PairType <- as.factor(df$PairType)
        df$Condition <- as.factor(df$Condition)
        df$StimulusGroup <- as.factor(df$StimulusGroup)
        df$Subject <- as.factor(df$Subject)
        df$Trial <- as.factor(df$Trial)

        model_full <- lmer(y ~ Condition + PairType + StimulusGroup + (1 | Subject), data=df, REML=FALSE)

        singular <- isSingular(model_full)
        if (singular) warning("Singular fit detected")

        real_pairtype_coef <- fixef(model_full)["PairType1"]

        ci <- confint(model_full, parm="PairType1", method="Wald")
        ci_low  <- as.numeric(ci[1])
        ci_high <- as.numeric(ci[2])

        N_PERMUTATIONS <- 3000
        perm_pairtype_coefs <- numeric(N_PERMUTATIONS)
        fail_count <- 0

        for (i in 1:N_PERMUTATIONS) {
            perm_df <- df
            for (subj in unique(df$Subject)) {
                for (cond in unique(df$Condition)) {
                    for (grp in unique(df$StimulusGroup)) {
                        idx <- which(df$Subject == subj & df$Condition == cond & df$StimulusGroup == grp)
                        if (length(idx) > 1) {
                            perm_df$PairType[idx] <- sample(df$PairType[idx])
                        }
                    }
                }
            }
            perm_df$PairType <- factor(perm_df$PairType, levels=c("0", "1"))
            perm_model <- tryCatch(
                lmer(y ~ Condition + PairType + StimulusGroup + (1 | Subject), data=perm_df, REML=FALSE),
                error = function(e) { fail_count <<- fail_count + 1; return(NA) }
            )
            if (is.na(perm_model)[1]) next
            perm_pairtype_coefs[i] <- fixef(perm_model)["PairType1"]
        }

        perm_pairtype_coefs <- perm_pairtype_coefs[!is.na(perm_pairtype_coefs)]
        p_value_pairtype_perm <- mean(perm_pairtype_coefs >= real_pairtype_coef)

        list(
            intercept=as.numeric(fixef(model_full)["(Intercept)"]),
            pair=as.numeric(fixef(model_full)["PairType1"]),
            condition2=as.numeric(fixef(model_full)["Condition2"]),
            stimgroup2=as.numeric(fixef(model_full)["StimulusGroup2"]),
            sigma_resid=as.numeric(sigma(model_full)),
            ci_low=ci_low,
            ci_high=ci_high,
            p_value_pairtype_perm=p_value_pairtype_perm,
            singular=singular,
            fail_count=fail_count
        )
        """

        r_results = ro.r(r_code)
        names = list(r_results.names)
        d = {n: r_results[i] for i, n in enumerate(names)}

        def scalar(x):
            try:
                return x[0]
            except (TypeError, IndexError):
                return x

        results = [
            region,
            float(scalar(d["intercept"])),
            float(scalar(d["pair"])),
            float(scalar(d["condition2"])),
            float(scalar(d["stimgroup2"])),
            float(scalar(d["sigma_resid"])),
            float(scalar(d["ci_low"])),
            float(scalar(d["ci_high"])),
            float(scalar(d["p_value_pairtype_perm"])),
            bool(scalar(d["singular"])),
            int(scalar(d["fail_count"])),
            int(n_excluded_subjects),
        ]
        print(f"Process {process_id} - Completed region {region}")
        return results

    except Exception as e:
        print(f"Process {process_id} - Error processing region {region}: {e}")
        return None


def run_analysis(realpair, crosspairs, num_cores=10,
                 exclude_extreme_subjects=False, subj_z_thresh=3.0):
    print(f"Running with {num_cores} cores | exclude_extreme_subjects={exclude_extreme_subjects}")

    pool = mp.Pool(processes=num_cores)
    process_func = partial(process_region, realpair=realpair, crosspairs=crosspairs,
                           exclude_extreme_subjects=exclude_extreme_subjects,
                           subj_z_thresh=subj_z_thresh)
    results = pool.map(process_func, range(400))
    pool.close()
    pool.join()

    results_list = [r for r in results if r is not None]

    df_results = pd.DataFrame(results_list, columns=[
        "Brain_Region", "Intercept", "Pair_Effect",
        "Condition_Effect", "StimulusGroup_Effect", "sigma_resid",
        "PairType_CI_low", "PairType_CI_high",
        "PairType_p_perm", "Singular_Fit", "Fail_Count",
        "N_Excluded_Subjects",
    ])

    df_results["d_like"] = df_results["Pair_Effect"] / df_results["sigma_resid"]
    df_results["PairType_beta_raw"] = df_results["Pair_Effect"] / 10.0
    df_results["PairType_CI_low_raw"] = df_results["PairType_CI_low"] / 10.0
    df_results["PairType_CI_high_raw"] = df_results["PairType_CI_high"] / 10.0

    return df_results


# ======================================================================
# usage: run twice and compare
# ======================================================================

# MAIN (identical to original)
# results_main = run_analysis(np.array(realpair), np.array(crosspairs),
#                             exclude_extreme_subjects=False)

# ROBUSTNESS (drop between-subject extremes)
results = run_analysis(np.array(realpair), np.array(crosspairs),
                       exclude_extreme_subjects=EXCLUDE_EXTREME_SUBJECTS,
                       subj_z_thresh=SUBJECT_Z_THRESH)

Running with 10 cores | exclude_extreme_subjects=True
Process 299997 - Processing region 0
Process 299998 - Processing region 10
Process 299999 - Processing region 20
Process 300000 - Processing region 30
Process 300001 - Processing region 40
Process 300002 - Processing region 50
Process 300003 - Processing region 60
Process 300004 - Processing region 70
Process 300005 - Processing region 80
Process 300006 - Processing region 90


R[write to console]: In addition: 
R[write to console]: There were 50 or more warnings (use warnings() to see the first 50)
R[write to console]: 



Process 300000 - Completed region 30
Process 300000 - Processing region 31


R[write to console]: In addition: 
R[write to console]: There were 50 or more warnings (use warnings() to see the first 50)
R[write to console]: 



Process 299998 - Completed region 10
Process 299998 - Processing region 11


...

Process 300002 - Completed region 389


R[write to console]: In addition: 
R[write to console]: There were 50 or more warnings (use warnings() to see the first 50)
R[write to console]: 



Process 299999 - Completed region 369


R[write to console]: In addition: 
R[write to console]: There were 50 or more warnings (use warnings() to see the first 50)
R[write to console]: 



Process 300004 - Completed region 399


In [23]:
from statsmodels.stats.multitest import multipletests

# 进行 FDR 校正（对所有脑区的 Time_Condition_p 进行校正）
results["PairType_p_perm_FDR"] = multipletests(results["PairType_p_perm"], method="fdr_bh")[1]
significant_regions1 = results[
    (results["PairType_p_perm_FDR"] < 0.05)]

print(significant_regions1)
# 提取符合条件的脑区编号
significant_region_numbers = significant_regions1["Brain_Region"].tolist()

# 输出脑区编号
print(significant_region_numbers)

     Brain_Region                 Intercept            Pair_Effect  \
3               3     [-0.8512745761712716]  [0.37060990964602947]   
36             36     [-0.6391077260446814]  [0.31281662874671406]   
40             40     [-0.5796542351506541]  [0.34911912663916495]   
54             54     [-1.0293698420502573]  [0.24102008214759688]   
58             58     [-1.7261096470778294]  [0.25143861949670354]   
59             59    [0.003089334508161923]  [0.20083864658482312]   
78             78     [0.17126783887239602]  [0.18757199977700173]   
79             79     [0.23765078635660364]   [0.2290666822929024]   
82             82  [-0.0033333365896475327]  [0.18177546440720582]   
100           100     [-0.1930058686979455]  [0.41447979612017866]   
106           106    [-0.04539568100435506]  [0.21702312604866578]   
139           139    [-0.14735729767131064]   [0.4028746395102664]   
140           140     [0.07990859367319221]  [0.45328166548578147]   
141           141   

In [16]:
# 保存为 CSV 文件
results.to_csv("sireal_260718.csv", index=False)

print("Results saved to 'sireal_0526.csv'")

Results saved to 'sireal_0526.csv'
